In [1]:
from pathlib import Path
import polars as pl

import numpy as np
from scipy.stats import spearmanr, kendalltau

In [2]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent

print(f"Folder where group folder should be: {BASE}")
print(f"Current working directory (where table will be saved): {CWD}")

Folder where group folder should be: /data/users/bdupin/datasets-organelle-igr
Current working directory (where table will be saved): /data/users/bdupin/datasets-organelle-igr/code


In [3]:
groups = [
    "fungi_mit",
    "metazoans_mit",
    "plants_mit",
    "plants_plt",
    "protists_mit",
    "protists_plt",
    "green_algae_mit",
    "green_algae_plt",
]

label_map = {
    "fungi_mit": "Fungi (mitochondria)",
    "green_algae_mit": "Green algae (mitochondria)",
    "metazoans_mit": "Metazoans (mitochondria)",
    "plants_mit": "Plants (mitochondria)",
    "protists_mit": "Protists (mitochondria)",
    "green_algae_plt": "Green algae (plastid)",
    "plants_plt": "Plants (plastid)",
    "protists_plt": "Protists (plastid)",
}

polarity_3bin = {
    "++": "same",
    "--": "same",
    "+-": "opposite",
    "-+": "opposite"
    }

polarity_2bin = {
    "++": "same",
    "--": "same",
    "+-": "opposite",
    "-+": "opposite",
}

MIN_IGR = 40
TAIL_FRACTION = 0.1

## S1

In [ ]:
global_true = 0
global_false = 0
global_total = 0

support_info = {}

for g in groups:
    igs = pl.read_csv(BASE / g / "summary_igs_intergenic.tsv", separator="\t")

    igs = igs.with_columns(
        pl.col("Polarity")
        .replace_strict(polarity_map)
        .cast(pl.Categorical)
        .alias("polarity_bin")
    )

    med = igs.group_by(["AN", "polarity_bin"]).agg(
        pl.col("Length").median().alias("median_bp")
    )

    wide = med.pivot(values="median_bp", index="AN", on="polarity_bin")
    wide = wide.with_columns((pl.col("opposite") > pl.col("same")).alias("support_opp"))

    group_true = wide.filter(pl.col("support_opp")).height
    group_false = wide.filter(~pl.col("support_opp")).height

    global_true += group_true
    global_false += group_false
    global_total += group_true + group_false

    support_info[g] = {
        "true": group_true,
        "false": group_false,
        "total_genomes": group_true + group_false,
    }

    print(f"Within Group {g}, out of {group_true + group_false} genomes:")
    print(
        f" Percent  True: {group_true / (group_true + group_false) * 100:.2f}% ({group_true})"
    )
    print(
        f" Percent False: {group_false / (group_true + group_false) * 100:.2f}% ({group_false})"
    )
    print("")

print(f"Overall Results, out of {global_total} genomes:")
print(
    f" Percent  True: {global_true / (global_true + global_false) * 100:.2f}% ({global_true})"
)
print(
    f" Percent False: {global_false / (global_true + global_false) * 100:.2f}% ({global_false})"
)

### Length-tail enrichment analysis

In [ ]:
def genome_enrichment_stats(df_g: pl.DataFrame, tail_frac: float) -> dict:
    n = df_g.height

    # Count polarities
    n_opp = df_g.filter(pl.col("polarity_bin") == "opposite").height
    n_same = df_g.filter(pl.col("polarity_bin") == "same").height

    # Δlog10 (opp - same) - computed using median(log10(length))
    median_opp = (
        df_g.filter(pl.col("polarity_bin") == "opposite")
        .select(pl.col("Length").log10().median())
        .item()
    )
    median_same = (
        df_g.filter(pl.col("polarity_bin") == "same")
        .select(pl.col("Length").log10().median())
        .item()
    )

    delta_log10 = median_opp - median_same
    fold_change = 10**delta_log10

    tail_size = int(round(tail_frac * n))

    # Sort by length and get top/bottom tails
    df_sorted = df_g.sort("Length", descending=True)
    top_tail = df_sorted.head(tail_size)
    bottom_tail = df_sorted.tail(tail_size)

    # Calculate fractions
    frac_opp_all = n_opp / n
    if frac_opp_all == 0:
        raise ValueError(
            f"Fraction of opposite polarity is zero, thefore AN {df_g['AN'][0]} is monopolar!"
        )

    frac_opp_top = (
        top_tail.filter(pl.col("polarity_bin") == "opposite").height / tail_size
    )

    frac_opp_bottom = (
        bottom_tail.filter(pl.col("polarity_bin") == "opposite").height / tail_size
    )

    # Enrichment ratios
    enrich_top = frac_opp_top / frac_opp_all
    depletion_botton = frac_opp_bottom / frac_opp_all

    return {
        "AN": df_g["AN"][0],
        "n_igr": n,
        "n_opp": n_opp,
        "n_same": n_same,
        "tail_size": tail_size,
        "frac_opp_all": frac_opp_all,
        "frac_opp_top": frac_opp_top,
        "frac_opp_bottom": frac_opp_bottom,
        "enrich_top": enrich_top,
        "depletion_botton": depletion_botton,
        "delta_log10": delta_log10,
        "fold_change": fold_change,
    }

In [ ]:
def genome_enrichment_stats(df_g: pl.DataFrame, tail_frac: float) -> dict:
    n = df_g.height

    # Count polarities
    n_opp = df_g.filter(pl.col("polarity_bin") == "opposite").height
    n_same = df_g.filter(pl.col("polarity_bin") == "same").height

    # Δlog10 (opp - same) - computed using median(log10(length))
    median_opp = (
        df_g.filter(pl.col("polarity_bin") == "opposite")
        .select(pl.col("Length").log10().median())
        .item()
    )
    median_same = (
        df_g.filter(pl.col("polarity_bin") == "same")
        .select(pl.col("Length").log10().median())
        .item()
    )

    delta_log10 = median_opp - median_same
    fold_change = 10**delta_log10

    tail_size = int(round(tail_frac * n))

    # Sort by length and get top/bottom tails
    df_sorted = df_g.sort("Length", descending=True)
    top_tail = df_sorted.head(tail_size)
    bottom_tail = df_sorted.tail(tail_size)

    # Calculate fractions
    frac_opp_all = n_opp / n
    if frac_opp_all == 0:
        raise ValueError(
            f"Fraction of opposite polarity is zero, thefore AN {df_g['AN'][0]} is monopolar!"
        )

    frac_opp_top = (
        top_tail.filter(pl.col("polarity_bin") == "opposite").height / tail_size
    )

    frac_opp_bottom = (
        bottom_tail.filter(pl.col("polarity_bin") == "opposite").height / tail_size
    )

    # Enrichment ratios
    enrich_top = frac_opp_top / frac_opp_all
    depletion_botton = frac_opp_bottom / frac_opp_all

    return {
        "AN": df_g["AN"][0],
        "n_igr": n,
        "n_opp": n_opp,
        "n_same": n_same,
        "tail_size": tail_size,
        "frac_opp_all": frac_opp_all,
        "frac_opp_top": frac_opp_top,
        "frac_opp_bottom": frac_opp_bottom,
        "enrich_top": enrich_top,
        "depletion_botton": depletion_botton,
        "delta_log10": delta_log10,
        "fold_change": fold_change,
    }

In [ ]:
enrich_path = BASE / "code" / "enrich_data"
enrich_path.mkdir(exist_ok=True)
group_enrich = {}
enrich_for_table = {}
global_over_40_igr = 0

for g in groups:
    print(f"Processing group: {g}")
    igs = pl.scan_csv(BASE / g / "summary_igs_intergenic.tsv", separator="\t")

    igs = igs.with_columns(
        pl.col("Polarity")
        .replace_strict(polarity_map)
        .cast(pl.Categorical)
        .alias("polarity_bin")
    ).collect()

    an_counts = igs.group_by("AN").agg(pl.len().alias("count"))
    valid_ans = an_counts.filter(pl.col("count") >= MIN_IGR)["AN"].to_list()
    global_over_40_igr += len(valid_ans)
    igs = igs.filter(pl.col("AN").is_in(valid_ans))

    rows = []
    for subset in igs.partition_by("AN", maintain_order=True):
        data = genome_enrichment_stats(
            subset,
            tail_frac=TAIL_FRACTION,
        )
        rows.append(data)

    per_genome = pl.DataFrame([r for r in rows if r])

    # Sort by AN and round float columns
    per_genome = per_genome.sort("AN").with_columns(pl.col(pl.Float64).round(3))

    # ties (i.e., 1.0) are considered not enriched/depleted, so we use > and < to create boolean columns
    per_genome = per_genome.with_columns(
        (pl.col("enrich_top") > 1.0).alias("top_enriched"),
        (pl.col("depletion_botton") < 1.0).alias("bottom_depleted"),
    )

    file_out = enrich_path / f"{g}_enrichment.tsv"
    per_genome.write_csv(file_out, separator="\t")
    per_genome = per_genome.with_columns(pl.lit(g).alias("group"))
    group_enrich[g] = per_genome

    n_genomes = per_genome.height
    n_top_enriched = per_genome.filter(pl.col("top_enriched")).height
    n_bottom_depleted = per_genome.filter(pl.col("bottom_depleted")).height

    enrich_for_table[g] = {
        "genomes_over_40_igr": n_genomes,
        "top_enriched_pct": round((n_top_enriched / n_genomes) * 100, 1),
        "top_enriched_n": n_top_enriched,
        "bottom_depleted_pct": round((n_bottom_depleted / n_genomes) * 100, 1),
        "bottom_depleted_n": n_bottom_depleted,
    }

    print(f" Results for {len(per_genome)} genomes written to {file_out}")
    print()

In [ ]:
df_all = pl.concat(group_enrich.values())
df_all = df_all.select(
    ["AN", "group"] + [col for col in df_all.columns if col not in ["AN", "group"]]
)
df_all = df_all.sort(["group", "AN"])
df_all.write_csv(enrich_path / "all_groups_enrichment.tsv", separator="\t")

In [ ]:
rows_s1 = []
total_genomes_over_40_igr = 0

for g in groups:
    info = support_info[g]
    enrich = enrich_for_table[g]
    total_genomes_over_40_igr += enrich["genomes_over_40_igr"]

    rows_s1.append(
        {
            "Taxonomic group and organelle type": label_map[g],
            "Total Genomes": info["total_genomes"],
            "% med(opposite) > med(same)": round(
                (info["true"] / info["total_genomes"]) * 100, 1
            ),
            "Genomes with >= 40 IGRs": enrich["genomes_over_40_igr"],
            "% Top Enriched (count)": f"{enrich['top_enriched_pct']}% ({enrich['top_enriched_n']})",
            "% Bottom Depleted (count)": f"{enrich['bottom_depleted_pct']}% ({enrich['bottom_depleted_n']})",
        }
    )
rows_s1.append(
    {
        "Taxonomic group and organelle type": "Overall",
        "Total Genomes": global_total,
        "% med(opposite) > med(same)": round((global_true / global_total) * 100, 1),
        "Genomes with >= 40 IGRs": global_over_40_igr,
        "% Top Enriched (count)": "-",
        "% Bottom Depleted (count)": "-",
    }
)

s1 = pl.DataFrame(rows_s1)
s1.write_csv(BASE / "code" / "supplemental_table1.tsv", separator="\t")

## S2

In [15]:
name_map = {
    "m_both": "both",
    "m_type": "igr_flanking_type",
    "m_len": "igr_len"}

def parse_kfold_block(text):
    lines = text.splitlines()
    in_block = False
    out = {}
 
    for line in lines:
        ls = line.strip()
        if ls.startswith("--- K-fold"):
            in_block = True
            continue
        if in_block:
            if ls.startswith("elpd_diff"):
                continue  # header row
            parts = ls.split()
            if len(parts) == 3 and parts[0] in name_map:
                model, elpd_diff, se_diff = parts
                out[name_map[model]] = (float(elpd_diff), float(se_diff))
            elif ls == "" or ls.startswith("Interpreting"):
                break
 
    if len(out) != 3:
        raise ValueError(f"Expected 3 models in K-fold block, found {len(out)}: {out}")
    return out

In [ ]:
rows_s2 = []

model_key_map = {
    "gene_length": "igr_len", 
    "gene_type": "igr_flanking_type", 
    "both": "both"}

for g in groups:
    gdir = BASE / g
    
    brms_pol = gdir / "brms_polarity" / "brms_results_row.tsv"
    brms_type_length_dir = gdir / "brms_type_length"
    brms_type_length = brms_type_length_dir / "brms_results_row.tsv"
    brms_type_length_txt = brms_type_length_dir / "brms_result.txt"
 
    tsv_polarity = pl.read_csv(brms_pol, separator="\t")
    tsv_type_length = pl.read_csv(brms_type_length, separator="\t")
    kfold = parse_kfold_block(brms_type_length_txt.read_text())
 
    label = label_map[g]
 
    # ---- base (polarity-only) model: single row ----
    polarity = tsv_polarity.row(0, named=True)
    rows_s2.append(
        {
            "group": label,
            "model": "polarity (base)",
            "N_regions": polarity["N_regions"],
            "N_taxa": polarity["N_taxa"],
            "fold_conv": polarity["fold_convergent_over_same"],
            "fold_conv_lo": polarity["fold_convergent_lo"],
            "fold_conv_hi": polarity["fold_convergent_hi"],
            "fold_div": polarity["fold_divergent_over_same"],
            "fold_div_lo": polarity["fold_divergent_lo"],
            "fold_div_hi": polarity["fold_divergent_hi"],
            "kfold_elpd_diff": None,
            "kfold_se_diff": None,
        }
    )
 
    # ---- len / type / both models: one row each ----
    for pred_type, model_name in model_key_map.items():
        r = tsv_type_length.filter(pl.col("predictor_type") == pred_type).row(0, named=True)
        elpd_diff, se_diff = kfold[model_name]
        rows_s2.append(
            {
                "group": label,
                "model": model_name,
                "N_regions": r["N_regions"],
                "N_taxa": r["N_taxa"],
                "fold_conv": r["fold_convergent_over_same"],
                "fold_conv_lo": r["fold_convergent_lo"],
                "fold_conv_hi": r["fold_convergent_hi"],
                "fold_div": r["fold_divergent_over_same"],
                "fold_div_lo": r["fold_divergent_lo"],
                "fold_div_hi": r["fold_divergent_hi"],
                "kfold_elpd_diff": elpd_diff,
                "kfold_se_diff": se_diff,
            }
        )
 
s2 = pl.DataFrame(rows_s2)

float_cols = [col for col in s2.columns if s2[col].dtype in [pl.Float32, pl.Float64]]
s2 = s2.with_columns([pl.col(col).round(2) for col in float_cols])

s2.write_csv(BASE / "code" / "supplemental_table2.tsv", separator="\t")

## S3

In [4]:
def spearman_ci(x, y, n_boot, rng):
    """Spearman rho with a percentile bootstrap 95% CI."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    n = len(x)
 
    rho0, _ = spearmanr(x, y)
 
    boot_rhos = np.empty(n_boot)
    idx = np.arange(n)
    for b in range(n_boot):
        samp = rng.choice(idx, size=n, replace=True)
        r, _ = spearmanr(x[samp], y[samp])
        boot_rhos[b] = r
 
    lo, hi = np.percentile(boot_rhos, [2.5, 97.5])
    return rho0, lo, hi, n

def kendall_ci(x, y, n_boot, rng):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    n = len(x)
 
    tau0, _ = kendalltau(x, y)
 
    boot_taus = np.empty(n_boot)
    idx = np.arange(n)
    for b in range(n_boot):
        samp = rng.choice(idx, size=n, replace=True)
        t, _ = kendalltau(x[samp], y[samp])
        boot_taus[b] = t
 
    lo, hi = np.percentile(boot_taus, [2.5, 97.5])
    return tau0, lo, hi, n

In [ ]:
rows = []
MIN_PAIRS = 5
N_BOOT = 2000
SEED = 42
rng = np.random.default_rng(SEED)
 
for g in groups:
    tsv = pl.read_csv(BASE / g / f"{g}.tsv", separator="\t")
    igs = pl.read_csv(BASE / g / f"{g}_summary_igr.tsv", separator="\t")
 
    igs = igs.join(
        tsv.select(["AN", "Genome_length"]),
        on="AN",
        how="left",
        validate="m:1",
    )
 
    igs = igs.with_columns(
        pl.col("Polarity").replace_strict(polarity_2bin).alias("polarity_2bin")
    )
 
    per_genome = igs.group_by("AN").agg(
        pl.col("Genome_length").first().alias("genome_size_bp"),
        pl.len().alias("n_pairs"),
        (pl.col("polarity_2bin") == "opp").sum().alias("n_opposite"),
        pl.col("Length").sum().alias("noncoding_bp"),
    ).sort("AN")
 
    per_genome = per_genome.with_columns(
        (pl.col("n_opposite") / pl.col("n_pairs")).alias("prevalence"),
        (pl.col("noncoding_bp") / pl.col("genome_size_bp")).alias("noncoding_fraction"),
    ).filter(pl.col("n_pairs") >= MIN_PAIRS)
 
    label = label_map[g]
 
    rho_size, lo_size, hi_size, n_size = spearman_ci(
        per_genome["prevalence"], per_genome["genome_size_bp"],
        n_boot=N_BOOT,
        rng=rng
    )
    rho_ncf, lo_ncf, hi_ncf, n_ncf = spearman_ci(
        per_genome["prevalence"], per_genome["noncoding_fraction"],
        n_boot=N_BOOT,
        rng=rng
    )

    # Kendall's tau-b cross-check (report alongside Spearman for any group
    # whose CI clears the +/-0.2 band, e.g. protists_plt genome size)
    tau_size, tlo_size, thi_size, _ = kendall_ci(
        per_genome["prevalence"], per_genome["genome_size_bp"],
        n_boot=N_BOOT,
        rng=rng
    )
    tau_ncf, tlo_ncf, thi_ncf, _ = kendall_ci(
        per_genome["prevalence"], per_genome["noncoding_fraction"],
        n_boot=N_BOOT,
        rng=rng
    )
 
    rows.append(
        {
            "group": label,
            "rho": rho_size,
            "lo_size": lo_size,
            "hi_size": hi_size,
            "n_size": n_size,
            "rho_ncf": rho_ncf,
            "lo_ncf": lo_ncf,
            "hi_ncf": hi_ncf,
            "n_ncf": n_ncf,
            "tau_size": tau_size,
            "tlo_size": tlo_size,
            "thi_size": thi_size,
            "tau_ncf": tau_ncf,
            "tlo_ncf": tlo_ncf,
            "thi_ncf": thi_ncf,
        }
    )
 
    print(f"{g} (N genomes = {per_genome.height}, min_pairs >= {MIN_PAIRS})")
    print(f"  rho (prevalence, genome size)        = {rho_size:.3f} ({lo_size:.3f},{hi_size:.3f})")
    print(f"  tau (prevalence, genome size)        = {tau_size:.3f} ({tlo_size:.3f},{thi_size:.3f})")
    print(f"  rho (prevalence, noncoding fraction) = {rho_ncf:.3f} ({lo_ncf:.3f},{hi_ncf:.3f})")
    print(f"  tau (prevalence, noncoding fraction) = {tau_ncf:.3f} ({tlo_ncf:.3f},{thi_ncf:.3f})\n")
 
s3 = pl.DataFrame(rows)
s3.write_csv(BASE / "code" / "supplemental_table3.tsv", separator="\t")

/tmp/ipykernel_2584744/3893446763.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho0, _ = spearmanr(x, y)
/tmp/ipykernel_2584744/3893446763.py:15: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = spearmanr(x[samp], y[samp])


fungi_mit (N genomes = 418, min_pairs >= 5)
  rho (prevalence, genome size)        = nan (nan,nan)
  tau (prevalence, genome size)        = nan (nan,nan)
  rho (prevalence, noncoding fraction) = nan (nan,nan)
  tau (prevalence, noncoding fraction) = nan (nan,nan)



KeyboardInterrupt: 

## S4

## S5

In [16]:
s5_data = []
for g in groups:
    tsv_path = BASE / g / f"{g}.tsv"
    brms_path = BASE / g / "brms_polarity" / "filtered_3bin.tsv"

    tsv = pl.read_csv(tsv_path, separator="\t")
    brms = pl.read_csv(brms_path, separator="\t")
    polarity = dict(brms["polarity_3bin"].value_counts().iter_rows())

    # median + IQR (Q1, Q3) on raw (non-transformed) IGR length, per polarity class
    len_stats = brms.group_by("polarity_3bin").agg(
        pl.col("igr_len").median().alias("median_bp"),
        pl.col("igr_len").quantile(0.25).alias("q1_bp"),
        pl.col("igr_len").quantile(0.75).alias("q3_bp"),
    )
    len_stats = {row["polarity_3bin"]: row for row in len_stats.iter_rows(named=True)}
    len_same = len_stats.get("same")
    len_conv = len_stats.get("conv")
    len_div = len_stats.get("div")

    def get_stat(bin_name, key, default=None):
        return len_stats.get(bin_name, {}).get(key, default)

    s5_data.append(
        {
            "Taxonomic group and organelle type": label_map[g],
            "Total Genomes": tsv["AN"].n_unique(),
            "Total IGRs": brms.height,
            "Unique Taxa": tsv["ncbi_taxid"].n_unique(),
            "Same polarity IGRs": polarity.get("same", 0),
            "Convergent polarity IGRs": polarity.get("conv", 0),
            "Divergent polarity IGRs": polarity.get("div", 0),
            "Same median (bp)": len_same.get("median_bp"),
            "Same IQR (bp)": f"{len_same.get("q1_bp"):.0f}-{len_same.get("q3_bp"):.0f}",
            "Convergent median (bp)": len_conv.get("median_bp"),
            "Convergent IQR (bp)": f"{len_conv.get("q1_bp"):.0f}-{len_conv.get("q3_bp"):.0f}",
            "Divergent median (bp)": len_div.get("median_bp"),
            "Divergent IQR (bp)": f"{len_div.get("q1_bp"):.0f}-{len_div.get("q3_bp"):.0f}",
        }
    )

s5 = pl.DataFrame(s5_data)
s5.write_csv(BASE / "code" / "supplemental_table5.tsv", separator="\t")